# Advanced Models — Train on Legacy, Test on Live Data

**Goal:** Compare 4 models: train on ALL legacy data (869 candles), test on collection.db.

**Models:**
1. **LogisticRegression** (~57 params)
2. **RandomForest** (~300K params)
3. **XGBoost** (~25K params)
4. **Deep Neural Network** (~141K params) — via `deep_neural_network.py`

In [ ]:
import json
import random
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from deep_neural_network import DeepNeuralNetworkRunner
from evaluator import Evaluator
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from technicals import compute_all
from tqdm import tqdm

random.seed(42)
np.random.seed(42)
MAX_BID = 0.85

## 1. Train all models on legacy data

In [ ]:
rows = []
with open(Path("../data/legacy_features.jsonl")) as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
NON_FEATURE_COLS = {
    "candle_id",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
df[feature_cols] = df[feature_cols].fillna(0.0)
scaler = StandardScaler()
X_train = scaler.fit_transform(df[feature_cols].values)
y_train = df["target"].values
print(f"Training on {df['candle_id'].nunique()} candles, {len(df):,} snapshots, {len(feature_cols)} features")

In [ ]:
import time as _time

# Model 1: LogisticRegression
print("Training LR...", flush=True)
t0 = _time.time()
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
print(f"LR: {lr.coef_.size + lr.intercept_.size} params ({_time.time() - t0:.1f}s)", flush=True)

# Model 2: RandomForest
print("Training RF...", flush=True)
t0 = _time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_params = sum(e.tree_.node_count for e in rf.estimators_) * 2
print(f"RF: {rf_params:,} params ({_time.time() - t0:.1f}s)", flush=True)

# Model 3: XGBoost
print("Training XGB...", flush=True)
t0 = _time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
xgb_params = sum(t.count("leaf") + t.count("yes") for t in xgb_model.get_booster().get_dump())
print(f"XGB: {xgb_params:,} params ({_time.time() - t0:.1f}s)", flush=True)

# Model 4: DeepNeuralNetwork
print("Setting up DNN...", flush=True)
t0 = _time.time()
X_dnn_tr, X_dnn_val, y_dnn_tr, y_dnn_val = train_test_split(
    X_train, y_train, test_size=0.05, random_state=42, stratify=y_train
)
print(f"  Split: train={X_dnn_tr.shape} val={X_dnn_val.shape} ({_time.time() - t0:.1f}s)", flush=True)
print("  Creating runner...", flush=True)
t0 = _time.time()
dnn = DeepNeuralNetworkRunner(X_dnn_tr, y_dnn_tr, X_dnn_val, y_dnn_val)
print(f"  Runner created ({_time.time() - t0:.1f}s)", flush=True)
print("  Calling setup()...", flush=True)
t0 = _time.time()
dnn.setup()
print(f"  Setup done ({_time.time() - t0:.1f}s)", flush=True)
print("  Calling train(epochs=3)...", flush=True)
t0 = _time.time()
dnn.train(epochs=20, patience=5)
print(f"  Training done ({_time.time() - t0:.1f}s)", flush=True)

## 2. Load live data and compute features

In [ ]:
conn = sqlite3.connect(str(Path("../data/collection.db")))
candles_db = pd.read_sql("SELECT * FROM candles ORDER BY start_time", conn)
snaps_db = pd.read_sql("SELECT * FROM snapshots ORDER BY candle_id, timestamp", conn)
conn.close()
new_candles = []
for _, cr in candles_db.iterrows():
    cid = cr["candle_id"]
    srows = snaps_db[snaps_db["candle_id"] == cid]
    snaps = []
    for _, s in srows.iterrows():
        ob = json.loads(s["orderbook_json"])
        snaps.append(
            {
                "timestamp": s["timestamp"],
                "tick_timestamp": s["tick_timestamp"],
                "elapsed_pct": s["elapsed_pct"],
                "btc_price": s["btc_price"],
                "btc_bid": s["btc_bid"],
                "btc_ask": s["btc_ask"],
                "up_bids": [ob["up_bids"][0]] if ob.get("up_bids") else [],
                "up_asks": [ob["up_asks"][0]] if ob.get("up_asks") else [],
                "down_bids": [ob["down_bids"][0]] if ob.get("down_bids") else [],
                "down_asks": [ob["down_asks"][0]] if ob.get("down_asks") else [],
                "up_last_trade": s["up_last_trade"],
                "down_last_trade": s["down_last_trade"],
                "market_volume": s["market_volume"],
            }
        )
    new_candles.append(
        {
            "candle_id": cid,
            "start_time": cr["start_time"],
            "end_time": cr["end_time"],
            "open": cr["open"],
            "high": cr["high"],
            "low": cr["low"],
            "close": cr["close"],
            "volume": cr["volume"],
            "outcome": cr["outcome"],
            "final_ret": cr["final_ret"],
            "snapshots": snaps,
        }
    )
new_rows = []
prior = []
for candle in tqdm(new_candles, desc="Computing features"):
    for si in range(len(candle["snapshots"])):
        snap = candle["snapshots"][si]
        indicators = compute_all(prior, candle["open"], candle["snapshots"][: si + 1])
        row = {
            "candle_id": candle["candle_id"],
            "elapsed_pct": snap["elapsed_pct"],
            "btc_price": snap["btc_price"],
            "up_best_ask": snap["up_asks"][0][0] if snap["up_asks"] else None,
            "down_best_ask": snap["down_asks"][0][0] if snap["down_asks"] else None,
            **indicators,
            "outcome": candle["outcome"],
        }
        new_rows.append(row)
    prior.append(candle)
df_live = pd.DataFrame(new_rows)
df_live["target"] = (df_live["outcome"] == "UP").astype(int)
df_live[feature_cols] = df_live[feature_cols].fillna(0.0)
X_live = scaler.transform(df_live[feature_cols].values)
y_live = df_live["target"].values
print(f"Live data: {len(df_live):,} snapshots from {df_live['candle_id'].nunique()} candles")

## 3. Evaluate — per-candle accuracy

In [ ]:
# Pre-compute ALL predictions in one call per model
print("Computing predictions...", flush=True)
all_probs = {
    "LogisticRegression": lr.predict_proba(X_live)[:, 1],
    "RandomForest": rf.predict_proba(X_live)[:, 1],
    "XGBoost": xgb_model.predict_proba(X_live)[:, 1],
    "DNN": dnn.predict_proba(X_live),
}
print("Done", flush=True)

models = [
    ("LogisticRegression", lr.coef_.size + lr.intercept_.size),
    ("RandomForest", rf_params),
    ("XGBoost", xgb_params),
    ("DNN", dnn.param_count()),
]

all_results = []
print(f"{'Model':<22} {'Params':>10} {'Correct':>8} {'Total':>6} {'Accuracy':>9}")
print("-" * 60)
for name, params in models:
    probs = all_probs[name]
    correct, total = 0, 0
    for cid in df_live["candle_id"].unique():
        cmask = df_live["candle_id"] == cid
        idx = cmask.values
        y_c = df_live.loc[cmask, "target"].values[0]
        vote = 1 if probs[idx].mean() >= 0.5 else 0
        total += 1
        if vote == y_c:
            correct += 1
    acc = correct / total
    all_results.append({"model": name, "params": params, "correct": correct, "total": total, "accuracy": acc})
    print(f"{name:<22} {params:>10,} {correct:>8} {total:>6} {acc * 100:>8.1f}%")

## 4. Snapshot-level evaluation

In [ ]:
for name, params in models:
    probs = all_probs[name]
    preds = (probs >= 0.5).astype(int)
    ev = Evaluator(y_live, preds, probs, title=f"{name} ({params:,} params)")
    ev.full_report()
    print()

## 5. Comparison chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = [r["model"] for r in all_results]
accs = [r["accuracy"] * 100 for r in all_results]
params = [r["params"] for r in all_results]
colors = ["#2ecc71" if a > 60 else "#f39c12" if a > 50 else "#e74c3c" for a in accs]
axes[0].barh(names, accs, color=colors, edgecolor="white")
axes[0].axvline(50, color="red", linestyle="--", alpha=0.3, label="random")
axes[0].set_xlabel("Per-Candle Accuracy (%)")
axes[0].set_title("Accuracy on Live Data")
for i, v in enumerate(accs):
    axes[0].text(v + 0.5, i, f"{v:.1f}%", va="center")
axes[0].legend()
axes[1].barh(names, params, color="mediumpurple", edgecolor="white")
axes[1].set_xscale("log")
axes[1].set_xlabel("Parameters (log scale)")
axes[1].set_title("Model Complexity")
for i, v in enumerate(params):
    axes[1].text(v * 1.2, i, f"{v:,}", va="center", fontsize=9)
plt.suptitle("Train: ALL legacy (869 candles) -> Test: collection.db", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Conclusion

### Per-Candle Accuracy (majority vote on unseen live data)

| Model | Parameters | Per-Candle Accuracy | Snapshot Accuracy | MSE | R² |
|-------|-----------|-------------------|-------------------|-----|-----|
| **RandomForest** | **688,276** | **83.2% (748/899)** | **73.3%** | **0.1754** | **29.7%** |
| LogisticRegression | 57 | 79.3% (713/899) | 71.8% | 0.1910 | 23.4% |
| XGBoost | 14,814 | 73.6% (662/899) | 68.2% | 0.2081 | 16.5% |
| DNN | 141,825 | 66.4% (597/899) | 62.2% | 0.2469 | 1.0% |

### The big surprise: **RandomForest beats everything at 83.2%**

In the earlier candle-level split (within legacy data), RandomForest showed 62% — barely above random. Now, trained on ALL legacy data and tested on truly unseen live data, it hits **83.2%**. That's 4 points above LogisticRegression (79.3%).

**Why it works now:**
- **More training data** — 869 candles (all legacy) vs 654 (80% split). RF needs volume.
- **Non-linear signal exists** — the 4% gap over LR proves there ARE feature interactions that trees capture but a linear model can't.

### Model-by-model

**LogisticRegression (79.3%, 57 params)** — Reliable baseline. Still strong, fast, interpretable.

**RandomForest (83.2%, 688K params)** — The winner. Balanced confusion matrix: 16,058 correct DOWN + 16,149 correct UP. Precision=77.1%, Recall=70.0%.

**XGBoost (73.6%, 14.8K params)** — Overfit to legacy patterns. Low recall (63.1%) — misses many UP candles.

**DNN (66.4%, 141K params)** — Worst performer. R²=1.0% (near-random probabilities). Predicts DOWN too aggressively. Needs more training epochs and more data.

### Updated recommendation

```
Primary model:  RandomForest (83.2% per-candle on live data)
Fallback:       LogisticRegression (79.3%, if RF too slow)
Entry trigger:  3-consecutive, start at elapsed >= 5%
Scaling:        2x entries (e5% + e50%)
```

### Next steps

1. **Retrain RF on legacy + new data** — 1,768 candles total
2. **Tune RF hyperparameters** — n_estimators, max_depth, min_samples_leaf
3. **Ensemble LR + RF** — average probabilities for potentially better accuracy
4. **Fix DNN** — more epochs, smaller architecture, or train on more data
